# The availability–expression lag


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hematteo/learning-to-read-out/blob/main/notebooks/01_availability_expression_lag.ipynb)

**Claim.** During pretraining, task-relevant signal appears in a language model's *hidden states*
before the model's own *readout* (the unembedding matrix $W_U$) expresses it in the output
logits. The model "knows" the answer before it can say it.

**How we test it.** For a two-choice task with single-token answers $(y^+, y^-)$, take the
hidden state $h_t$ that the model at pretraining step $t$ feeds into its readout, but score it
with the readout $W_U^s$ taken from a *different* step $s$:

$$\mathrm{margin}(t, s) \;=\; h_t \cdot W_U^s[y^+] \;-\; h_t \cdot W_U^s[y^-]$$

The diagonal $s{=}t$ is what the model actually does. If an early hidden state $h_t$ scores far
better under a *later* readout ($s \gg t$) than under its own, the signal was **available** in
$h_t$ but not yet **expressed** by $W_U^t$ — that gap is the lag.

**What this notebook does, live:**

1. builds four contrastive task families (subject–verb agreement, induction, indirect-object
   identification, country→capital facts),
2. downloads six public [Pythia-160M](https://huggingface.co/EleutherAI/pythia-160m) pretraining
   checkpoints (~2.3 GB, cached after the first run),
3. computes the full margin grid over (hidden step $t$, readout step $s$), corrects for gauge
   drift between checkpoints, and plots the lag — clearest for factual recall at this scale.

**Runtime:** ~10–15 min end-to-end on a free Colab CPU runtime (a GPU speeds up the forward
passes but is optional). Everything is seeded and CPU-reproducible.

> This is a small live version of `experiments/probes/contrastive_readout_swap/` in the
> [repository](https://github.com/hematteo/learning-to-read-out), which runs Pythia-1B over the
> full 32-checkpoint grid with gauge-alignment ladders and matched-distractor controls (thesis
> figure `fig:lr-lag`). Companion notebook:
> [`02_wu_trajectory_crosscoders.ipynb`](02_wu_trajectory_crosscoders.ipynb) shows *how* the
> readout forms.

In [ ]:
# Setup: locate (or clone) the repository and import its probe utilities.
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hematteo/learning-to-read-out"


def find_repo_root():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "probes" / "readout_swap.py").exists():
            return cand
    return None


ROOT = find_repo_root()
if ROOT is None:  # fresh Colab runtime -> clone
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    ROOT = Path.cwd() / "learning-to-read-out"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:  # match the repo's floor (pyproject.toml: transformers>=5.0,<6)
    _tv = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    _tv = "0.0"
if tuple(int(x) for x in _tv.split(".")[:2]) < (5, 0):
    %pip install -q "transformers>=5,<6"

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.core.model_specs import DEFAULT_STEPS_32
from src.core.repro import seed_everything
from src.probes import contrastive_tasks as CT
from src.probes import readout_swap as RS

seed_everything(0)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"repo:   {ROOT}")
print(f"torch {torch.__version__} | transformers {importlib.metadata.version('transformers')} | device: {DEVICE}")

## 1 · Six checkpoints, four task families

Pythia publishes every pretraining checkpoint as a HuggingFace revision (`step256`, `step512`,
…). We use six steps spanning the run — a subset of the paper's canonical 32-step schedule
(`src/core/model_specs.py:DEFAULT_STEPS_32`). Add steps to `STEPS` below if you want a denser
grid; each one costs ~375 MB of download.

The task families come from `src/probes/contrastive_tasks.py`. Each example is a prompt plus a
*single-token* answer pair $(y^+, y^-)$ — single-token so that one readout row per answer is all
that's compared. Builders enforce the paper's filtering: both answers tokenize to one token with
matched leading space, and (given a reference $W_U$) the two rows' norms match within
$|\log r| < 0.5$, so a margin can't come from one row simply being longer.

| family | example prompt | $y^+$ / $y^-$ |
|---|---|---|
| `sva` | `The keys to the cabinet` | ` are` / ` is` |
| `induction` | ` apple river apple river apple` | ` river` / matched random token |
| `ioi` | `Mark and Lisa went to the store. Mark gave the book to` | ` Lisa` / ` Mark` |
| `relational_facts` | `The capital of France is` | ` Paris` / matched capital |

In [ ]:
MODEL = "EleutherAI/pythia-160m"

# Six of the paper's 32 canonical snapshot steps (final step 143000 = the released model).
STEPS = [256, 512, 1000, 2000, 8000, 143000]
assert all(s in DEFAULT_STEPS_32 for s in STEPS)

# family -> (builder, n_max). Yields are smaller than n_max after single-token filtering.
FAMILIES = {
    "sva": (CT.build_sva, 96),
    "induction": (CT.build_induction, 64),
    "ioi": (CT.build_ioi, 64),
    "relational_facts": (CT.build_relational_facts, 48),
}

In [ ]:
# The FINAL checkpoint's W_U is the norm-matching reference for task filtering,
# and its tokenizer defines what counts as a single token.
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Everything extracted from a checkpoint is cached on disk, so re-running the
# notebook (or restarting the kernel) skips both downloads and forward passes.
CACHE_DIR = ROOT / "local_snapshots" / "swap_demo" / MODEL.split("/")[-1]  # gitignored
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def load_model_at(step):
    return (
        AutoModelForCausalLM.from_pretrained(MODEL, revision=f"step{step}", dtype=torch.float32)
        .to(DEVICE)
        .eval()
    )


def wu_at(step):
    """Readout W_U at `step`: (V, d_model) fp32 on CPU, disk-cached."""
    p = CACHE_DIR / f"wu_step{step}.pt"
    if not p.exists():
        model = load_model_at(step)
        torch.save(model.embed_out.weight.detach().float().cpu().clone(), p)
        del model
    return torch.load(p, map_location="cpu", weights_only=True)


W_U_final = wu_at(STEPS[-1])
print("W_U:", tuple(W_U_final.shape), "(V, d_model)")

examples = {}
for name, (builder, n_max) in FAMILIES.items():
    examples[name] = builder(tokenizer, n_max=n_max, W_U_for_norm_match=W_U_final)
    ex = examples[name][0]
    print(
        f"{name:<18} n={len(examples[name]):>3}   e.g. {ex.prompt!r} "
        f"-> {tokenizer.decode([ex.y_plus])!r} vs {tokenizer.decode([ex.y_minus])!r}"
    )

## 2 · Extract hidden states and readouts at every step

For each checkpoint we keep two things:

- $W_U^s$ — its readout (`embed_out.weight`, the unembedding);
- $h_t$ — for every prompt, the residual-stream vector that enters the readout at the **final
  prompt token** (captured with a forward-pre-hook on `embed_out`, exactly as in
  `src/probes/readout_swap.py:extract_final_hidden`).

This is the slow cell on a first run: it forwards ~250 short prompts through each of the six
checkpoints, downloading each checkpoint once. Everything lands in the disk cache, so re-runs
of the cell — including after a kernel restart — are seconds.

In [ ]:
import time


def h_path(family, step):
    # n in the key invalidates the cache if you change a family's n_max above.
    return CACHE_DIR / f"h_{family}_n{len(examples[family])}_step{step}.pt"


W_U = {}  # step -> (V, d_model)
H = {}  # (family, step) -> (N, d_model)

for step in STEPS:
    t0 = time.time()
    missing = [f for f in examples if not h_path(f, step).exists()]
    if missing:
        model = load_model_at(step)
        torch.save(model.embed_out.weight.detach().float().cpu().clone(), CACHE_DIR / f"wu_step{step}.pt")
        for name in missing:
            h = RS.extract_final_hidden(model, [e.prompt_ids for e in examples[name]], device=DEVICE)
            torch.save(h, h_path(name, step))
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    W_U[step] = wu_at(step)
    for name in examples:
        H[(name, step)] = torch.load(h_path(name, step), map_location="cpu", weights_only=True)
    print(f"step {step:>6}: W_U + hidden states ready ({time.time() - t0:.0f}s)")

## 3 · When does the model *express* each task?

First the diagonal: native accuracy, scoring $h_t$ with the model's own readout $W_U^t$.
Accuracy = fraction of examples with $\mathrm{margin} > 0$; chance is 0.5.

In [ ]:
T = len(STEPS)
fams = list(FAMILIES)
y_pm = {
    f: (
        torch.tensor([e.y_plus for e in examples[f]]),
        torch.tensor([e.y_minus for e in examples[f]]),
    )
    for f in fams
}

# acc[f][i, j] = accuracy of hidden step STEPS[i] under readout step STEPS[j].
acc = {f: np.zeros((T, T)) for f in fams}
for f in fams:
    y_p, y_m = y_pm[f]
    for i, t in enumerate(STEPS):
        for j, s in enumerate(STEPS):
            cell = RS.evaluate_swap_cell(H[(f, t)], W_U[s], W_U[t], y_p, y_m, alignment="none", device="cpu")
            acc[f][i, j] = cell["accuracy"]

fig, ax = plt.subplots(figsize=(6.5, 3.6))
xs = np.arange(T)
for f in fams:
    ax.plot(xs, np.diag(acc[f]), "o-", label=f)
ax.axhline(0.5, ls=":", c="gray", lw=1)
ax.set_xticks(xs, [f"{s:,}" for s in STEPS])
ax.set_xlabel("pretraining step $t$")
ax.set_ylabel("accuracy, native readout $(t,t)$")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.set_title("Expression: what the model actually says")
plt.show()

## 4 · The readout-swap grid

Now every cell $(t, s)$: hidden states from step $t$, readout from step $s$, no other change.
The diagonal is the previous plot. **Read along a row**: fixing an early hidden state $h_t$ and
moving right (later readouts) shows what later readouts can already decode from it.

In [ ]:
fig, axes = plt.subplots(1, len(fams), figsize=(4.1 * len(fams), 3.9), constrained_layout=True)
for ax, f in zip(np.atleast_1d(axes), fams):
    im = ax.imshow(acc[f], vmin=0.0, vmax=1.0, cmap="RdBu_r", origin="upper")
    for i in range(T):
        for j in range(T):
            ax.text(j, i, f"{acc[f][i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(acc[f][i, j] - 0.5) > 0.3 else "black")
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, ec="k", lw=1.5))
    ax.set_xticks(range(T), [f"{s//1000}k" if s >= 1000 else s for s in STEPS], fontsize=8)
    ax.set_yticks(range(T), [f"{s//1000}k" if s >= 1000 else s for s in STEPS], fontsize=8)
    ax.set_xlabel("readout step $s$")
    ax.set_title(f, fontsize=10)
axes[0].set_ylabel("hidden step $t$")
fig.colorbar(im, ax=axes, shrink=0.8, label="accuracy (chance = 0.5)")
plt.show()

Black boxes mark the native diagonal. Three things to notice:

1. **Expression onset** along the diagonal — each family switches on at its own step.
2. **The converse control** in the lower-left: late hidden states scored by early readouts
   collapse toward chance — and for `relational_facts`, far *below* chance: the early readout's
   answer rows are actively misordered, not merely uninformative.
3. **Hints of rescue** above-right of the diagonal (e.g. the `sva` and `induction` rows just at
   onset) — an early hidden state scoring better under a later readout.

The raw grid *understates* that third effect, though, for a mundane reason: $W_U$ at step 1k and
step 143k live in different **gauges** (overall scale, per-row norms, orientation), so dotting an
early $h_t$ against raw late rows mixes "better-organized directions" with mere basis drift.

## 5 · Availability vs. expression, gauge-corrected

The paper handles this with an alignment ladder
(`src/probes/readout_swap.py:align_readout`): before scoring, align $W_U^s$ to $W_U^t$'s gauge —
re-center + globally rescale (`scale`), transplant per-row norms (`row_norm`), or apply the
orthogonal **Procrustes** rotation (`procrustes`). All are fit on the full matrices, never on
task answers, so none can inject task knowledge.

The summary plot: per hidden step $t$, native accuracy against accuracy under the **final
readout after Procrustes alignment**. The shaded gap is the availability–expression lag.
Error bars are 95% binomial normal-approximation intervals — $n$ per family is small (it's a
demo), so read exact values loosely; the paper works with per-example margins and paired
bootstraps instead.

In [ ]:
# Accuracy of every hidden step under the Procrustes-aligned final readout.
acc_final_proc = {}
for f in fams:
    y_p, y_m = y_pm[f]
    acc_final_proc[f] = np.array(
        [
            RS.evaluate_swap_cell(
                H[(f, t)], W_U[STEPS[-1]], W_U[t], y_p, y_m, alignment="procrustes", device="cpu"
            )["accuracy"]
            for t in STEPS
        ]
    )

def ci95(p, n):
    # 95% binomial normal-approximation interval (collapses at p=0 or 1).
    return 1.96 * np.sqrt(p * (1 - p) / n)


fig, axes = plt.subplots(1, len(fams), figsize=(3.4 * len(fams), 3.1), sharey=True, constrained_layout=True)
xs = np.arange(T)
for ax, f in zip(np.atleast_1d(axes), fams):
    n = len(examples[f])
    native = np.diag(acc[f])
    rescued = acc_final_proc[f]
    ax.errorbar(xs, native, yerr=ci95(native, n), fmt="o-", color="C0", capsize=2,
                label="native readout $W_U^t$")
    ax.errorbar(xs, rescued, yerr=ci95(rescued, n), fmt="s-", color="C1", capsize=2,
                label="final readout, Procrustes-aligned")
    ax.fill_between(xs, native, rescued, where=rescued >= native, color="C1", alpha=0.15)
    ax.axhline(0.5, ls=":", c="gray", lw=1)
    ax.set_xticks(xs, [f"{s//1000}k" if s >= 1000 else s for s in STEPS], fontsize=8)
    ax.set_xlabel("hidden step $t$")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{f}  (n={n})", fontsize=10)
axes[0].set_ylabel("accuracy")
axes[0].legend(fontsize=7, loc="lower right")
fig.suptitle("Available in $h_t$ (orange) before expressed by $W_U^t$ (blue)", y=1.06)
plt.show()

Read panel by panel — at this small scale the lag is **family-heterogeneous**, and that is
the honest result:

- **`relational_facts`** is the vivid case: at step 1k the native readout is at chance (0.5)
  while the aligned final readout decodes the same hidden states at ~0.7; at step 2k, ~0.73
  native vs ~1.0 rescued. The model *represents* " Paris" before its readout can say it.
- **`sva`** shows a smaller gap right at expression onset (step 1k).
- **`induction` and `ioi`** show no lag at 160M: availability and expression rise together —
  whatever those circuits need from the readout is in place by the time the hidden-state signal
  exists. At 1B, with the full 32-step grid and per-example statistics, the paper finds rescue
  across a broader family set; treat the 160M demo as qualitative.

A rotation fit on the *whole* matrices being this load-bearing is itself a finding: much of what
separates an early readout from a late one is organization of the same subspace, not new rows.

## 6 · The full alignment ladder

One family under every rung — `none` is the most conservative score, `procrustes` the most
gauge-corrected; the native diagonal is dashed. If a rescue only appeared at one rung you would
worry; here the ladder brackets it.

In [ ]:
CHECK_FAMILY = "relational_facts"  # <-- try the others too

y_p, y_m = y_pm[CHECK_FAMILY]
fig, ax = plt.subplots(figsize=(5.6, 3.4))
for al in ("none", "scale", "row_norm", "procrustes"):
    curve = [
        RS.evaluate_swap_cell(
            H[(CHECK_FAMILY, t)], W_U[STEPS[-1]], W_U[t], y_p, y_m, alignment=al, device="cpu"
        )["accuracy"]
        for t in STEPS
    ]
    ax.plot(np.arange(T), curve, "o-", label=f"final readout, align={al}")
ax.plot(np.arange(T), np.diag(acc[CHECK_FAMILY]), "k--", marker=".", label="native readout")
ax.axhline(0.5, ls=":", c="gray", lw=1)
ax.set_xticks(np.arange(T), [f"{s//1000}k" if s >= 1000 else s for s in STEPS], fontsize=8)
ax.set_xlabel("hidden step $t$")
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
ax.set_title(f"{CHECK_FAMILY}: rescue under gauge alignment")
ax.legend(fontsize=8)
plt.show()

## What we simplified — and where the real pipeline lives

This notebook trades completeness for runtime. The paper's version differs in scale and rigor:

- **Model / grid:** Pythia-1B (and OLMo-2-7B), full 32×32 checkpoint grid vs. our 160M, 6×6.
- **Families:** the full registry in `src/probes/contrastive_tasks.py` adds harder SVA,
  role-balanced IOI, numeric comparison, and six benchmark-derived families (PIQA, ARC, SciQ,
  LAMBADA, WinoGrande).
- **Controls:** matched-random distractors, label-permutation controls for MCQ families,
  per-example margins with paired bootstraps, and the full alignment ladder on every cell.

Reproduce the real thing from the repository root:

```bash
uv run python experiments/probes/contrastive_readout_swap/scripts/build_task_datasets.py --model pythia-1b
uv run python experiments/probes/contrastive_readout_swap/scripts/run_swap_grid.py \
    --model pythia-1b --datasets-dir results/experiments/probes/contrastive_readout_swap/datasets/pythia-1b
```

See [`docs/REPRODUCE.md`](../docs/REPRODUCE.md) for the figure→script map (this experiment backs
thesis figure `fig:lr-lag`), and [`experiments.yaml`](../experiments.yaml) for the full manifest.

**Next:** the lag says early hidden states wait on the readout. *How* does the readout then get
built? [`02_wu_trajectory_crosscoders.ipynb`](02_wu_trajectory_crosscoders.ipynb) trains the
paper's instrument — a parameter-trajectory crosscoder on $W_U$ snapshots — and watches sparse
readout features form.

**Citation:** see [`CITATION.cff`](../CITATION.cff) (thesis Chapter 4; paper link added on release).